# Biscuit King AI Snack Finder
### IS215 Digital Business — Technologies and Transformation
This notebook demonstrates the AI-powered chatbot component of the Biscuit King digital transformation prototype. The chatbot uses the Anthropic Claude API with a curated product knowledge base to provide personalised snack recommendations.

### Step 1: Import relevant libraries

In [ ]:
import pandas as pd
import json
import os
from anthropic import Anthropic

### Step 2: Load and explore product data
Load the Biscuit King product catalogue from Excel and examine the dataset using pandas.

In [ ]:
df = pd.read_excel('biscuit_king_product_template.xlsx')

# Clean column names
df.columns = df.columns.str.strip()
df = df.rename(columns={
    'ID': 'id',
    'Product Name': 'name',
    'Image Filename': 'image',
    'Category': 'category',
    'Price (SGD)': 'price',
    'Price Unit': 'price_unit',
    'Flavor Profile': 'flavor_profile',
    'Era': 'era',
    'Occasions': 'occasions',
    'Description': 'description'
})

print(f"Total products: {len(df)}")
df.head(10)

In [ ]:
# Product distribution by category
print("Products per category:")
print(df['category'].value_counts())
print(f"\nPrice range: ${df['price'].min():.2f} — ${df['price'].max():.2f}")
print(f"\nCategories: {df['category'].unique().tolist()}")

In [ ]:
# Parse pipe-delimited fields into lists
for col in ['flavor_profile', 'era', 'occasions']:
    df[col] = df[col].apply(lambda x: [i.strip() for i in str(x).split('|')] if pd.notna(x) else [])

# Example: show eras for first product
print(f"Product: {df.iloc[0]['name']}")
print(f"Eras: {df.iloc[0]['era']}")
print(f"Occasions: {df.iloc[0]['occasions']}")

### Step 3: Build the product knowledge base
Convert the pandas DataFrame into a text-based knowledge base that will be injected into the AI system prompt. This ensures the chatbot ONLY references real products with accurate prices.

In [ ]:
def products_to_knowledge_base(df):
    """Convert DataFrame to text knowledge base for the AI system prompt."""
    products_text = ""
    for _, row in df.iterrows():
        products_text += f"""
- **{row['name']}** (ID: {row['id']})
  Category: {row['category']}
  Price: ${row['price']:.2f} per {row['price_unit']}
  Flavour: {', '.join(row['flavor_profile'])}
  Era: {', '.join(row['era'])}
  Occasions: {', '.join(row['occasions'])}
  Description: {row['description']}
"""
    return products_text

knowledge_base = products_to_knowledge_base(df)

# Preview first 500 chars
print(knowledge_base[:500])
print(f"\n... ({len(knowledge_base)} characters total)")

### Step 4: Build system prompt with hallucination guardrails
The system prompt defines the chatbot's persona, injects the product data, and includes strict rules to prevent the AI from inventing products, prices, or promotions that don't exist.

In [ ]:
system_prompt = f"""You are "Biscuit King", a friendly and knowledgeable snack shop assistant for Biscuit King — a traditional snack retailer at 130 Casuarina Road, Upper Thomson, Singapore.

## YOUR PERSONA
- Warm, enthusiastic, and genuinely passionate about traditional snacks
- You speak naturally and can use light Singlish where appropriate
- Keep responses concise: 2-4 sentences, then offer to help more

## YOUR KNOWLEDGE — PRODUCT CATALOGUE
You know EXACTLY {len(df)} products. Here is your COMPLETE catalogue — this is your ONLY source of truth:

{knowledge_base}

## STORE INFORMATION
- Address: 130 Casuarina Road, Singapore 579518
- Hours: Tuesday – Sunday, 11am – 10pm (Closed on Mondays)
- Phone: +65 6454 5938
- Mix & Match Tins: Choose up to 4 flavours per tin, from $30
- In-store pickup available

## HALLUCINATION PREVENTION RULES
1. ONLY recommend products listed above. NEVER invent product names or prices.
2. If asked about a product not in the catalogue, say so and suggest a real alternative.
3. ALWAYS include the exact price and unit when recommending.
4. NEVER make up promotions or discounts.
5. If unsure, say so honestly.
6. For non-snack topics, politely redirect to snacks.
"""

print(f"System prompt length: {len(system_prompt)} characters")

### Step 5: Initialize the Anthropic client and test the chatbot
Connect to the Claude API and test with sample queries that demonstrate the chatbot's recommendation capabilities.

In [ ]:
# API key loaded from environment variable (NEVER hardcode)
client = Anthropic(api_key=os.environ.get('ANTHROPIC_API_KEY'))
conversation_history = []

def chat(user_message):
    """Send a message to the chatbot and get a response."""
    conversation_history.append({"role": "user", "content": user_message})
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=500,
        system=system_prompt,
        messages=conversation_history
    )
    
    reply = response.content[0].text
    conversation_history.append({"role": "assistant", "content": reply})
    return reply

def reset_chat():
    """Clear conversation history for a fresh start."""
    conversation_history.clear()

### Step 6: Test — Era-based recommendation
Simulate a user asking for snacks from their childhood decade.

In [ ]:
reset_chat()
response = chat("I'm a 90s kid, what snacks should I get?")
print(f"User: I'm a 90s kid, what snacks should I get?\n")
print(f"Biscuit King: {response}")

### Step 7: Test — Gift tin recommendation within budget

In [ ]:
reset_chat()
response = chat("I want to build a gift tin under $50 for my grandmother. What do you recommend?")
print(f"User: I want to build a gift tin under $50 for my grandmother. What do you recommend?\n")
print(f"Biscuit King: {response}")

### Step 8: Test — Occasion-based recommendation

In [ ]:
reset_chat()
response = chat("What snacks are good for Chinese New Year?")
print(f"User: What snacks are good for Chinese New Year?\n")
print(f"Biscuit King: {response}")

### Step 9: Test — Hallucination guardrail
Test that the chatbot does NOT invent products. Asking for a product that doesn't exist should trigger a redirect to real products.

In [ ]:
reset_chat()
response = chat("Do you sell matcha biscuits?")
print(f"User: Do you sell matcha biscuits?\n")
print(f"Biscuit King: {response}")

In [ ]:
response = chat("What's the weather like today?")
print(f"User: What's the weather like today?\n")
print(f"Biscuit King: {response}")

### Step 10: Test — Conversation memory
Demonstrate that the chatbot remembers context within a conversation.

In [ ]:
reset_chat()
r1 = chat("I like sour snacks")
print(f"User: I like sour snacks")
print(f"Biscuit King: {r1}\n")

r2 = chat("Any of those good for kids' parties?")
print(f"User: Any of those good for kids' parties?")
print(f"Biscuit King: {r2}\n")

r3 = chat("How much would 3 of those cost?")
print(f"User: How much would 3 of those cost?")
print(f"Biscuit King: {r3}")

### Summary
This notebook demonstrates:
1. **Data processing** — Loading and parsing product data using pandas
2. **Knowledge base construction** — Converting structured data into an AI-readable format
3. **Hallucination prevention** — System prompt guardrails that restrict the AI to only reference real products with accurate prices
4. **Conversational AI** — Multi-turn conversations with context memory
5. **Use cases** — Era-based, occasion-based, budget-based, and flavour-based recommendations

The same Python logic (`chatbot.py`) powers the live chatbot on the Biscuit King website via a backend API route.